This notebook builds an updated dataset that preserves all CARAVAN data (with gauge precipitation) and appends additional precipitation variable based on MSWEP, CHIRPS and gauge precipitation.

In [4]:
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pickle

In [5]:
BACKUP_DIR = Path("./filtered_data_2/time_series")
DATA_DIR   = Path("./data/time_series")
MSWEP_DIR = Path("./mswep_precip_timeseries")
GAUGE_DIR = Path("./gauge_precip_timeseries")
CHIRPS_DIR = Path("./chirps_precip_timeseries")

In [7]:
PRECIP_VAR = "prcp_mm_day"
MSWEP_VAR = "prcp_mswep_mm_day"
GAUGE_VAR = "prcp_gauge_mm_day"
CHIRPS_VAR = "prcp_chirps_mm_day"
TIME_VAR = "date"

DATA_DIR.mkdir(parents=True, exist_ok=True)

for nc_in in BACKUP_DIR.glob("CAMELS_UY_*.nc"):
    basin_id = nc_in.stem
    print(f"\nProcessing basin: {basin_id}")

    nc_out = DATA_DIR / nc_in.name
    # csv_file = UYPRECIP_DIR / f"{basin_id}_precip.csv"
    mswep_csv_file = MSWEP_DIR / f"{basin_id}_precip.csv"
    gauge_csv_file = GAUGE_DIR / f"{basin_id}_precip.csv"
    chirps_csv_file = CHIRPS_DIR / f"{basin_id}_precip.csv"

    ds = xr.open_dataset(nc_in).load()

    if PRECIP_VAR not in ds or TIME_VAR not in ds.coords:
        print("  ⏭ Missing precip or time coord – skipping")
        ds.close()
        continue

    ds_time = pd.to_datetime(ds[TIME_VAR].values)

    # -------------------------------------------------
    # Helper function to standardize any CSV
    # -------------------------------------------------
    def load_precip_csv(path, date_col, precip_col):
        df = pd.read_csv(path)
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
        df = df.rename(columns={precip_col: "precip"})
        return df[["precip"]]

    # -------------------------------------------------
    # LOAD DATASETS
    # -------------------------------------------------

    # Gauge
    if not gauge_csv_file.exists():
        print("  ⏭ No gauge file")
        ds.close()
        continue

    df_gauge = load_precip_csv(
        gauge_csv_file,
        date_col="Fecha",
        precip_col="precip_gauge"
    )

    # MSWEP
    if not mswep_csv_file.exists():
        print("  ⏭ No MSWEP file")
        ds.close()
        continue

    df_mswep = load_precip_csv(
        mswep_csv_file,
        date_col="date",
        precip_col="precip_mm"
    )

    # CHIRPS (date is index column)
    if not chirps_csv_file.exists():
        print("  ⏭ No CHIRPS file")
        ds.close()
        continue

    df_chirps = pd.read_csv(chirps_csv_file, index_col=0)
    df_chirps.index = pd.to_datetime(df_chirps.index)
    df_chirps = df_chirps.rename(columns={"precipitation": "precip"})
    df_chirps = df_chirps[["precip"]]

    # -------------------------------------------------
    # ALIGN & ADD VARIABLES
    # -------------------------------------------------

    def add_product_to_ds(df, var_name):
        common_dates = ds_time.intersection(df.index)

        if len(common_dates) == 0:
            print(f"  ⏭ No overlap for {var_name}")
            return

        print(f"  ✔ Adding {var_name} for {len(common_dates)} days")

        new_var = xr.full_like(ds[PRECIP_VAR], np.nan)
        new_var.loc[{TIME_VAR: common_dates}] = (
            df.loc[common_dates, "precip"].values
        )

        ds[var_name] = new_var

    add_product_to_ds(df_gauge, GAUGE_VAR)
    add_product_to_ds(df_mswep, MSWEP_VAR)
    add_product_to_ds(df_chirps, CHIRPS_VAR)

    ds.attrs["precip_update"] = "Gauge, MSWEP and CHIRPS added where available"

    ds.to_netcdf(nc_out)
    ds.close()

print("\n✅ All basins processed successfully.")


Processing basin: CAMELS_UY_7
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing basin: CAMELS_UY_2
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing basin: CAMELS_UY_16
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing basin: CAMELS_UY_9
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing basin: CAMELS_UY_5
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing basin: CAMELS_UY_11
  ✔ Adding prcp_gauge_mm_day for 11322 days
  ✔ Adding prcp_mswep_mm_day for 11322 days
  ✔ Adding prcp_chirps_mm_day for 11322 days

Processing ba

In [8]:
CAMELS_UY_10=xr.open_dataset(DATA_DIR / "CAMELS_UY_10.nc")
CAMELS_UY_10

<xarray.Dataset> Size: 498kB
Dimensions:             (date: 11322)
Coordinates:
  * date                (date) datetime64[ns] 91kB 1989-01-01 ... 2019-12-31
Data variables:
    tmin_C              (date) float32 45kB ...
    tmax_C              (date) float32 45kB ...
    srad_W_m2           (date) float32 45kB ...
    prcp_mm_day         (date) float32 45kB ...
    QObs_mm_d           (date) float64 91kB ...
    prcp_gauge_mm_day   (date) float32 45kB ...
    prcp_mswep_mm_day   (date) float32 45kB ...
    prcp_chirps_mm_day  (date) float32 45kB ...
Attributes:
    precip_update:  Gauge, MSWEP and CHIRPS added where available